# G48 — Listener proxy vs true responder: groups, matching confidence, manual check (development; analysis only)

For every development MCIS the identities of clips I–III are rebuilt exactly as in RoleNet (ArcFace, average linkage,
cosine 0.45); A and the proxy L come from clip III. The true responder B is the dominant identity of clip IV (G44a
faces; analysis only, never an input), matched to the identity of clips I–III with the highest centroid cosine.

| Group | Rule (checked in this order) |
|---|---|
| `uncertain` | no clip-IV identity in ≥ 35% of its frames, or best cosine in [0.40, 0.55), or margin to the second-best identity < 0.05, or the best match is A |
| `B not seen` | best cosine < 0.40 |
| `B seen = proxy` | best cosine ≥ 0.55, margin ≥ 0.05, matched identity = proxy L |
| `B seen ≠ proxy` | same confidence, matched identity ≠ proxy L (or no proxy) |

Saved: `g48_groups.csv` (one row per MCIS: group, confidence values, whether B is visible in clip III, frames of B and
of L, and the G26/G44 rule's match for comparison) and a manual-check sheet: for 8 MCIS of each of `B seen ≠ proxy`,
`uncertain` and `B seen = proxy`, one image with B's clip-IV face, the matched clip I–III face, the proxy L and A in clip
III, and `g48_manual_check.csv` to fill in (`match_correct`, `proxy_correct`). Performance by group is computed
afterwards from the saved G41 / G44 / G43 predictions (no training here).

Inputs: `hi-ef-dataset` (video + annotation), `hi-ef-split`, `g8a-features`, the G44a output (`c4shard_*.pkl`). CPU.

In [ ]:
import os, glob, pickle, random
from collections import defaultdict
import numpy as np, pandas as pd, cv2
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.cluster import AgglomerativeClustering
from tqdm.auto import tqdm

DATASET_DIR = "/kaggle/input/datasets/ptrnghieu/hi-ef-dataset"
SPLIT_GLOB = "/kaggle/input/**/source_folder_split_seed42.csv"
G8A_GLOB = "/kaggle/input/**/shard_*.pkl"          # clips I-III (does not match c4shard_*)
C4_GLOB = "/kaggle/input/**/c4shard_*.pkl"         # clip IV faces (G44a): analysis only
OUT_DIR = "/kaggle/working/g48"
SAME_PERSON_COS, DOMINANT_MIN_FRAC = 0.45, 0.25    # as RoleNet / G26 / G44
FRAC_SURE, COS_LOW, COS_SURE, MARGIN = 0.35, 0.40, 0.55, 0.05
N_CHECK, CHECK_SEED = 8, 0
DEBUG_PER_EPISODE = None
os.makedirs(OUT_DIR, exist_ok=True)


def one(pattern, what):
    hits = sorted(glob.glob(pattern, recursive=True))
    assert hits, f"{what} not found ({pattern})"
    return hits[0]


sp = pd.read_csv(one(SPLIT_GLOB, "split csv"), dtype=str)
DEV = sp[sp.split.isin(['train', 'val'])].reset_index(drop=True)
if DEBUG_PER_EPISODE:
    DEV = DEV.groupby('source_folder').head(DEBUG_PER_EPISODE).reset_index(drop=True)
assert 'test' not in set(DEV.split)
need = set(DEV[['clip1', 'clip2', 'clip3']].values.ravel())
G8, G84 = {}, {}
for f in sorted(glob.glob(G8A_GLOB, recursive=True)):
    G8.update({k: v for k, v in pickle.load(open(f, 'rb')).items() if k in need})
for f in sorted(glob.glob(C4_GLOB, recursive=True)):
    G84.update({k: v for k, v in pickle.load(open(f, 'rb')).items() if k in set(DEV.clip4)})
miss, miss4 = need - set(G8), set(DEV.clip4) - set(G84)
assert not miss, f"{len(miss)} clips missing from g8a-features"
assert len(miss4) <= 0.01 * len(DEV), f"{len(miss4)} clip-IV records missing: attach the G44a output"
roots = sorted(glob.glob(os.path.join(DATASET_DIR, "*", "Hi-EF")))
VIDEO_ROOTS = [os.path.join(r, "video") for r in roots if os.path.isdir(os.path.join(r, "video"))]
print(f"development MCIS {len(DEV)} | video roots {VIDEO_ROOTS}")

In [ ]:
unit = lambda v: v / (np.linalg.norm(v) + 1e-9)


def cluster(E):
    return (AgglomerativeClustering(n_clusters=None, metric='cosine', linkage='average',
                                    distance_threshold=1 - SAME_PERSON_COS).fit_predict(E) if len(E) > 1 else np.zeros(len(E), int))


ROWS, KEEP = [], {}
for n, row in enumerate(tqdm(DEV.itertuples(), total=len(DEV), desc='groups')):
    cl = [row.clip1, row.clip2, row.clip3]
    items = [(k, j) for k, c in enumerate(cl) for j in range(len(G8[c]['faces']))]
    E = np.stack([G8[cl[k]]['faces'][j]['arc'] for k, j in items]).astype(np.float32) if items else np.zeros((0, 512), np.float32)
    lab = cluster(E)
    frames = defaultdict(set)
    for (k, j), p in zip(items, lab):
        frames[(k, p)].add(G8[cl[k]]['faces'][j]['frame'])
    ids3 = sorted({p for (k, p) in frames if k == 2}, key=lambda p: -len(frames[(2, p)]))
    A = ids3[0] if ids3 else None
    L = ids3[1] if len(ids3) > 1 else None
    # clip IV: dominant identity
    f4 = G84.get(row.clip4, {}).get('faces', [])
    frac, best, second, bp, c4lab = 0.0, -1.0, -1.0, None, None
    if f4:
        E4 = np.stack([d['arc'] for d in f4]).astype(np.float32)
        l4 = cluster(E4)
        fr4 = defaultdict(set)
        for d, p in zip(f4, l4):
            fr4[p].add(d['frame'])
        dom = max(fr4, key=lambda p: len(fr4[p]))
        frac = len(fr4[dom]) / max(G84[row.clip4]['meta']['n_sampled'], 1)
        c4lab = (l4, dom)
        if items:
            c4 = unit(E4[l4 == dom].mean(0))
            sims = sorted(((float(unit(E[lab == p].mean(0)) @ c4), p) for p in set(lab.tolist())), reverse=True)
            best, bp = sims[0]
            second = sims[1][0] if len(sims) > 1 else -1.0
    margin = best - second
    if frac < FRAC_SURE or (COS_LOW <= best < COS_SURE) or (best >= COS_LOW and margin < MARGIN) or (best >= COS_LOW and bp == A):
        group = 'uncertain'
    elif best < COS_LOW:
        group = 'B not seen'
    else:
        group = 'B seen = proxy' if bp == L else 'B seen ≠ proxy'
    rule_B = bp if (frac >= DOMINANT_MIN_FRAC and best >= SAME_PERSON_COS and bp != A) else None   # G26 / G44 rule
    nB3 = len(frames[(2, bp)]) if bp is not None and best >= COS_LOW else 0
    ROWS.append({'sample_id': row.sample_id, 'group': group, 'clip4_dominant_frac': frac, 'best_cos': best,
                 'second_cos': second, 'margin': margin, 'best_is_A': bp == A and bp is not None,
                 'has_proxy': L is not None, 'n_ids_III': len(ids3),
                 'B_frames_III': nB3, 'B_frames_I_II': (len(frames[(0, bp)]) + len(frames[(1, bp)])) if bp is not None and best >= COS_LOW else 0,
                 'L_frames_III': len(frames[(2, L)]) if L is not None else 0,
                 'rule_matched': rule_B is not None, 'rule_B_is_proxy': rule_B is not None and rule_B == L})
    KEEP[row.sample_id] = dict(cl=cl, items=items, lab=lab, A=A, L=L, bp=bp, f4=f4, c4lab=c4lab, clip4=row.clip4)
G = pd.DataFrame(ROWS)
G.to_csv(f"{OUT_DIR}/g48_groups.csv", index=False)
print(G.group.value_counts().to_string())
print("\nB visible in clip III, by group:")
print(G.assign(B_in_III=G.B_frames_III > 0).groupby('group').B_in_III.mean().round(3).to_string())
print(f"\nG26/G44 rule: matched {G.rule_matched.mean() * 100:.1f}% | proxy = rule's B among MCIS with a proxy and a match "
      f"{G[G.has_proxy & G.rule_matched].rule_B_is_proxy.mean() * 100:.1f}%")
print(pd.crosstab(G.group, G.rule_matched).to_string())
assert G.group.isin(['uncertain', 'B not seen', 'B seen = proxy', 'B seen ≠ proxy']).all()

In [ ]:
# ---- manual-check sheet: B's clip-IV face, the matched clip I-III face, the proxy L and A in clip III
def video_path(clip):
    ep, num = clip.split('/')
    for root in VIDEO_ROOTS:
        for ext in ('.mp4', '.avi', '.mkv', '.mov'):
            p = os.path.join(root, ep, num + ext)
            if os.path.exists(p):
                return p
    return None


def crop(clip, face, pad=0.35):
    if clip is None or face is None:                 # no such person (e.g. no proxy L) -> grey placeholder
        return np.full((96, 96, 3), 230, np.uint8)
    p = video_path(clip)
    if p is None:
        return np.full((96, 96, 3), 230, np.uint8)
    cap = cv2.VideoCapture(p)
    cap.set(cv2.CAP_PROP_POS_MSEC, max(face['t'], 0) * 1000)
    ok, fr = cap.read()
    cap.release()
    if not ok:
        return np.full((96, 96, 3), 230, np.uint8)
    x1, y1, x2, y2 = [int(v) for v in face['box']]
    w, h = x2 - x1, y2 - y1
    x1, y1 = max(0, int(x1 - pad * w)), max(0, int(y1 - pad * h))
    x2, y2 = min(fr.shape[1], int(x2 + pad * w)), min(fr.shape[0], int(y2 + pad * h))
    return cv2.cvtColor(fr[y1:y2, x1:x2], cv2.COLOR_BGR2RGB) if x2 > x1 and y2 > y1 else np.full((96, 96, 3), 230, np.uint8)


def biggest(faces):
    return max(faces, key=lambda d: (d['box'][2] - d['box'][0]) * (d['box'][3] - d['box'][1])) if faces else None


def faces_of(K, p, clips=(0, 1, 2)):
    return [G8[K['cl'][k]]['faces'][j] for (k, j), q in zip(K['items'], K['lab']) if q == p and k in clips]


rng = random.Random(CHECK_SEED)
CHECK = []
for grp in ['B seen ≠ proxy', 'uncertain', 'B seen = proxy']:
    ids = sorted(G[G.group == grp].sample_id)
    CHECK += [(grp, s) for s in rng.sample(ids, min(N_CHECK, len(ids)))]
SHEET = []
for i, (grp, s) in enumerate(CHECK):
    K, r = KEEP[s], G[G.sample_id == s].iloc[0]
    fb4 = None
    if K['c4lab'] is not None:
        l4, dom = K['c4lab']
        fb4 = biggest([d for d, q in zip(K['f4'], l4) if q == dom])
    panels = [('B in clip IV (target, analysis only)', crop(K['clip4'], fb4)),
              (f"matched I-III identity (cos {r.best_cos:.2f}, margin {r.margin:.2f})",
               crop(K['cl'][2], biggest(faces_of(K, K['bp'], (2,)))) if K['bp'] is not None and biggest(faces_of(K, K['bp'], (2,)))
               else crop(K['cl'][1], biggest(faces_of(K, K['bp'], (0, 1)))) if K['bp'] is not None and biggest(faces_of(K, K['bp'], (0, 1)))
               else crop(None, None)),
              ('proxy L, clip III', crop(K['cl'][2], biggest(faces_of(K, K['L'], (2,)))) if K['L'] is not None else crop(None, None)),
              ('speaker A, clip III', crop(K['cl'][2], biggest(faces_of(K, K['A'], (2,)))) if K['A'] is not None else crop(None, None))]
    fig, axs = plt.subplots(1, 4, figsize=(10, 2.9))
    for ax, (t, im) in zip(axs, panels):
        ax.imshow(im); ax.set_title(t, fontsize=7); ax.axis('off')
    fig.suptitle(f"[{i:02d}] {s} | group: {grp} | clip-IV dominant frac {r.clip4_dominant_frac:.2f}", fontsize=8)
    fig.tight_layout(); fig.savefig(f"{OUT_DIR}/check_{i:02d}_{s}.png", dpi=110); plt.close(fig)
    SHEET.append({'idx': i, 'sample_id': s, 'group': grp, 'best_cos': r.best_cos, 'margin': r.margin,
                  'clip4_dominant_frac': r.clip4_dominant_frac, 'match_correct': '', 'proxy_correct': '', 'note': ''})
pd.DataFrame(SHEET).to_csv(f"{OUT_DIR}/g48_manual_check.csv", index=False)
print(f"saved {len(SHEET)} check images and g48_manual_check.csv in {OUT_DIR}")
print("fill in match_correct (is the matched I-III face the same person as B in clip IV?) and proxy_correct "
      "(is the proxy L the same person as B?) with yes / no / unclear")